## Load latest headlines

In [1]:
# ========== Bootstrap: ensure required Python packages are present =========
import importlib, subprocess, sys
from typing import Optional

def _ensure(pkg_name: str, import_name: Optional[str] = None):
    try:
        importlib.import_module(import_name or pkg_name)
    except ModuleNotFoundError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg_name])
    finally:
        globals()[import_name or pkg_name] = importlib.import_module(import_name or pkg_name)

_ensure('pandas')
print('All dependencies ready.\n')


All dependencies ready.



In [2]:
from pathlib import Path
archive = Path('archive')
archive.mkdir(exist_ok=True)
import pandas as pd
latest = pd.read_csv('../headlines/latest.csv')
latest.head()

,pubdate,source,title,link
0,2026-11-04-00-56-05 +0000,bbc,When will we know the results?,https://www.bbc.co.uk/news/articles/cq20321y2n...
1,2026-11-04-00-54-00 +0000,wsj,Gold Edges Higher Amid Cooling Oil Prices,https://www.wsj.com/finance/commodities-future...
2,2026-11-04-00-53-55 +0000,nypost,Dramatic Election Day odds shift changes every...,https://nypost.com/2026/11/03/us-news/election...
3,2026-11-04-00-53-11 +0000,chitri,2026 Illinois Election Day results,https://www.chicagotribune.com/2026/11/03/2026...
4,2026-11-04-00-50-05 +0000,startribune,Live: Polls close soon in Minnesota amid repor...,https://www.startribune.com/live-minnesota-202...


## Calculate word frequencies

In [3]:
import re
from collections import Counter
from datetime import datetime

with open('exclude.txt') as f:
    stop_words = set(w.strip() for w in f if w.strip())
words = re.findall(r'[A-Za-z]+', ' '.join(latest['title']).lower())
filtered = [w for w in words if w not in stop_words and len(w) > 1]
counts = Counter(filtered)
score_df = (
    pd.DataFrame(counts.items(), columns=['word','score'])
    .sort_values('score', ascending=False)
)
score_df[['score','word']].to_csv('scores.csv', index=False)
timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')
score_df[['score','word']].to_csv(archive / f'scores-{timestamp}.csv', index=False)
score_df.head()


/tmp/ipykernel_2447/2220774760.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  timestamp = datetime.utcnow().strftime('%Y-%m-%d-%H-00-00')


,word,score
11,election,50
162,trump,32
12,day,28
135,midterm,20
2,results,19


## Rank headlines by score

In [4]:
word_scores = dict(score_df[['word','score']].values)
latest['score'] = latest['title'].apply(
    lambda t: sum(
        word_scores.get(w.lower(), 0)
        for w in re.findall(r'[A-Za-z]+', t)
        if len(w) > 1
    )
)
ranked = latest.sort_values('score', ascending=False)
ranked[['score','pubdate','source','title','link']].to_csv('rank.csv', index=False)
ranked[['score','pubdate','source','title','link']].to_csv(archive / f'rank-{timestamp}.csv', index=False)
ranked.head()


,pubdate,source,title,link,score
72,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...,212
334,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...,168
154,2026-11-03-20-32-42 +0000,chitri,Lake County Election Day turnout strong and st...,https://www.chicagotribune.com/2026/11/03/lake...,161
304,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...,158
58,2026-11-03-23-04-42 +0000,nypost,Dem pollster dishes on which California electi...,https://nypost.com/2026/11/03/us-news/pollster...,153


## Select top headlines

In [5]:
with open('exclude_sources.txt') as f:
    excluded_sources = {
        line.strip()
        for line in f
        if line.strip() and not line.strip().startswith('#')
    }

top_count = 10
top_rows = []
working = word_scores.copy()
remaining = latest.copy()

while len(top_rows) < top_count and not remaining.empty:
    scored_remaining = remaining.assign(
        score=remaining['title'].apply(
            lambda t: sum(
                working.get(w.lower(), 0)
                for w in re.findall(r'[A-Za-z]+', t)
                if len(w) > 1
            )
        )
    )
    ranked_loop = scored_remaining.sort_values('score', ascending=False)

    if ranked_loop.empty:
        break

    top_story = ranked_loop.iloc[0]
    remaining = remaining.drop(top_story.name)

    if top_story['source'] in excluded_sources:
        continue

    words = set(re.findall(r'[A-Za-z]+', top_story['title'].lower()))
    for word in words:
        working.pop(word, None)

    top_rows.append(top_story[['score', 'pubdate', 'source', 'title', 'link']])

top_df = pd.DataFrame(top_rows)
top_df.to_csv('top.csv', index=False)
top_df.to_csv(archive / f'top-{timestamp}.csv', index=False)
top_df


,score,pubdate,source,title,link
72,212,2026-11-03-22-49-00 +0000,wsj,Republicans and Democrats face off in a high-s...,https://www.wsj.com/politics/policy/democrats-...
154,161,2026-11-03-20-32-42 +0000,chitri,Lake County Election Day turnout strong and st...,https://www.chicagotribune.com/2026/11/03/lake...
389,119,2026-11-03-02-00-00 +0000,wsj,Tuesday’s midterm elections will offer clues t...,https://www.wsj.com/politics/elections/what-th...
53,71,2026-11-03-23-18-22 +0000,nypost,Midterms 2026 elections results map: Live upda...,https://nypost.com/2026/11/03/us-news/midterms...
334,59,2026-11-03-10-56-00 +0000,wsj,Trump surprises Republicans with an election-e...,https://www.wsj.com/politics/elections/control...
325,54,2026-11-03-11-00-00 +0000,latimes,Majority of California voters fear federal gov...,https://www.latimes.com/california/story/2026-...
331,47,2026-11-03-11-00-00 +0000,latimes,Hiltzik: Can AI bots become conscious? Why Mic...,https://www.latimes.com/business/story/2026-11...
319,46,2026-11-03-11-05-43 +0000,chitri,‘Absolutely astonishing’: Pastry fans line up ...,https://www.chicagotribune.com/2026/11/03/trit...
25,43,2026-11-04-00-10-35 +0000,nypost,"A-listers hit the polls for LA election, revea...",https://nypost.com/2026/11/03/us-news/hollywoo...
304,41,2026-11-03-11-57-00 +0000,wsj,"Follow the WSJ’s Election Day live coverage, a...",https://www.wsj.com/politics/elections/your-ho...


## Create JSON version of top headlines

In [6]:
import pandas as pd
pd.read_csv('top.csv').to_json('top.json', orient='records', indent=2)
